# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all data elements by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset metadata and create the dataset object
dataset = mlc.Dataset(croissant_url)

# Print dataset high-level metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}\nPublished: {dataset.metadata.datePublished}\nVersion: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. All entity lookups use their Croissant `@id` for precise referencing.

Let's inspect all record sets in the dataset, with their available fields.

In [ ]:
# Get all record sets by id
metadata = dataset.metadata
all_record_sets = metadata.recordSets

print(f"Record sets found ({len(all_record_sets)}):")
for rs in all_record_sets:
    print(f"- @id: {rs['@id']} | name: {rs['name']} | description: {rs.get('description', '')}")

    # List fields in the record set
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"    - field @id: {f['@id']} | name: {f['name']} | dataType: {f.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values discovered above.

**We'll extract the main protein identification record set (`@id` typically like `cr:ProteinRecordSet`) as an example.**

In [ ]:
# Fill in the record set @id from the overview: look for proteins table, e.g., 'cr:ProteinRecordSet'
# If uncertain, print all record set ids:
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print('Available record set @id\'s:', record_set_ids)
# For this dataset, we assume one main proteins table, e.g., 'cr:main'
main_record_set_id = record_set_ids[0] # Replace with actual record set @id if needed

# Load records from selected record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records with columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply representative data processing steps: filtering, normalization, categorization using field and record set `@id`s. 

**Example: Filter proteins with `PeptideCount` greater than 10, normalize `MolecularWeight`, and group by `SampleType` if present. (Replace with actual field @id as needed).**

In [ ]:
# Inspect available fields to choose appropriate @id
print('Columns:', df.columns.tolist())

# Assign field @id variables based on inspection above (update to match your dataset)
peptide_count_field = [col for col in df.columns if 'peptide' in col.lower() and 'count' in col.lower()][0] # e.g. 'cr:PeptideCount'
mw_field = [col for col in df.columns if 'molecular' in col.lower() and 'weight' in col.lower()][0] # e.g. 'cr:MolecularWeight'
sample_type_field = None
for col in df.columns:
    if 'sample' in col.lower() and 'type' in col.lower():
        sample_type_field = col

# Filter by PeptideCount > 10
threshold = 10
filtered_df = df[df[peptide_count_field] > threshold]
print(f"Filtered records with {peptide_count_field} > {threshold}:")
print(filtered_df.head())

# Normalize MolecularWeight
mw_norm_field = f"{mw_field}_normalized"
filtered_df[mw_norm_field] = (filtered_df[mw_field] - filtered_df[mw_field].mean()) / filtered_df[mw_field].std()
print(f"Normalized {mw_field} for filtered records:")
print(filtered_df[[mw_field, mw_norm_field]].head())

# Optional: group by sample type if available
if sample_type_field and sample_type_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(sample_type_field).mean(numeric_only=True)
    print(f"Grouped data by {sample_type_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships. We'll plot distribution of Molecular Weight for filtered proteins and, if available, the peptide count vs. normalized MW.

**Note**: This example uses field @id variables so it is robust to renaming.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of MolecularWeight
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[mw_field], bins=30, kde=True)
plt.title(f"Distribution of {mw_field}")
plt.xlabel(mw_field)
plt.show()

# PeptideCount vs Normalized MW scatter
plt.figure(figsize=(7, 5))
sns.scatterplot(x=peptide_count_field, y=mw_norm_field, data=filtered_df)
plt.title(f"{peptide_count_field} vs {mw_norm_field}")
plt.xlabel(peptide_count_field)
plt.ylabel(mw_norm_field)
plt.show()

## 6. Conclusion
- This notebook loaded and explored the FAIR² protein dataset using the Croissant schema and `mlcroissant`, referencing all structure by `@id`.
- We filtered proteins with high peptide counts, normalized molecular weights, and explored their distributions and relationships.
- See the [mlcroissant docs](https://mlcommons.github.io/croissant-python/latest/) and FAIR² documentation for advanced use and schema mapping.

**Continue to adapt the above code by referencing specific `@id` values in your workflows for full transparency and reproducibility.**